## Assign GRUPO_CAP
- some links did not have GRUPO_CAP assigned to them 
- assign GRUPO_CAP to important links according to their OSM classification
- leave GRUPO_CAP empty only on footways, steps, cycleways, etc.

In [18]:
import pandas as pd
import numpy as np
import win32com.client as com
import os
import ast

In [2]:
FOLDER = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
INSUMOS_IMEPLAN = r"Insumos IMEPLAN\Transporte Publico\FRECUENCIAS"
RED_FOLDER = r"Red Base GDL\RedBase Final\version completa"

In [3]:
#Red base GDL (con 2,203 zonas)
red_base = os.path.join(FOLDER, RED_FOLDER, "RedFinal - completa.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [31]:
# Read links from Visum
links = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "FromNode": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNode": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "TSysSet": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TSysSet")],
    "TypeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TypeNo")],
    "GRUPO_CAP": [i[1] for i in Visum.Net.Links.GetMultiAttValues("GRUPO_CAP")],
    "HIGHWAY": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")],
})

print(f"Total links: {len(links):,}")
print(f"Links with original Type 0 : {(links["TypeNo"]==0).sum():,}")

Total links: 605,828
Links with original Type 0 : 88,152


#### Original Link Types were __0 & 1__
- __those that keep being 0 or 1, Camilo couldn't assign a new link type because they didn't have GRUPO_CAP__
- 1's are mostly links that Johan created manually mostly BRT I assume
- 0's need special looking and a GRUPO_CAP based on their OSM HIGHWAY value

In [38]:
links_type0 = links[links["TypeNo"] == 0].copy()
print(links_type0["HIGHWAY"].value_counts().to_string())

HIGHWAY
residential                                    41176
tertiary                                       14167
                                               10586
secondary                                       5203
primary                                         3898
unclassified                                    2505
service                                         2372
living_street                                   1829
footway                                         1756
cycleway                                         739
trunk                                            707
primary_link                                     418
tertiary_link                                    389
path                                             377
track                                            339
secondary_link                                   281
trunk_link                                       275
pedestrian                                       269
motorway_link                         

In [26]:
highway_to_grupo_cap = {
    # Carretera / interurbana
    "motorway": "CARR",
    "trunk": "CARR",

    # Arteria urbana principal
    "primary": "ART",

    # Primaria menor / lateral
    "secondary": "MEN",

    # Colectora
    "tertiary": "COL",

    # Local
    "unclassified": "LOC",
    "residential": "LOC",
    "living_street": "LOC",
    "service": "LOC",

    # Acceso / rampa
    "motorway_link": "ACC",
    "trunk_link": "ACC",
    "primary_link": "ACC",
    "secondary_link": "ACC",
    "tertiary_link": "ACC",
}

# For when HIGHWAY is array
priority = [
    "motorway_link",
    "trunk_link",
    "primary_link",
    "secondary_link",
    "tertiary_link",
    
    "motorway",
    "trunk",
    "primary",
    "secondary",
    "tertiary",
    
    "unclassified",
    "residential",
    "living_street",
    "service",
]

In [32]:
def to_list(x):
    if pd.isna(x) or x == "":
        return []

    # Caso "['footway', 'residential']"
    if x.startswith("["):
        return ast.literal_eval(x)

    # Caso "residential"
    return [x]


links["HIGHWAY_LIST"] = links["HIGHWAY"].apply(to_list)

In [33]:
def get_grupo_cap(highways):
    for h in priority:
        if h in highways:
            return highway_to_grupo_cap[h]
    
    return ""

mask_type0 = (
    (links["TypeNo"] == 0)
    & (links["GRUPO_CAP"].isna() | links["GRUPO_CAP"].eq(""))
)

links.loc[mask_type0, "GRUPO_CAP"] = (
    links.loc[mask_type0, "HIGHWAY_LIST"]
    .apply(get_grupo_cap)
)

In [34]:
links.loc[
    mask_type0,
    ["HIGHWAY", "HIGHWAY_LIST", "GRUPO_CAP"]
].head(30)

,HIGHWAY,HIGHWAY_LIST,GRUPO_CAP
1,,[],
25,tertiary,[tertiary],COL
27,tertiary,[tertiary],COL
29,,[],
34,unclassified,[unclassified],LOC
35,unclassified,[unclassified],LOC
36,unclassified,[unclassified],LOC
37,unclassified,[unclassified],LOC
39,residential,[residential],LOC
40,unclassified,[unclassified],LOC


In [42]:
mask = (
    (links["TypeNo"] == 0)
    & (links["GRUPO_CAP"].isna() | links["GRUPO_CAP"].eq(""))
)
print(links.loc[mask, "HIGHWAY"].value_counts(dropna=False).to_string())

HIGHWAY
                                      10561
footway                                1756
cycleway                                739
path                                    377
track                                   339
pedestrian                              269
['footway', 'steps']                    107
steps                                    24
['track', 'path']                         4
['footway', 'path']                       3
['footway', 'path', 'steps']              2
ladder                                    2
['footway', 'pedestrian']                 2
busway                                    2
['footway', 'cycleway']                   2
['footway', 'pedestrian', 'steps']        1


In [47]:
links_visum = Visum.Net.Links

links = links.reset_index(drop=True)
links.index = links.index + 1

grupo_cap = list(zip(links.index, links["GRUPO_CAP"].astype(str)))

links_visum.SetMultiAttValues("GRUPO_CAP", grupo_cap)